```markdown
# Introdução prática à API do DeepSeek 🚀

Este notebook é um guia prático para iniciantes que desejam aprender a integrar e utilizar a API do DeepSeek em seus projetos. Aqui você aprenderá desde a configuração de credenciais seguras até a realização de chamadas de chat utilizando a biblioteca oficial da OpenAI e requisições HTTP diretas.

### Objetivos de Aprendizagem
* Entender como gerenciar e acessar chaves de API com segurança no Google Colab utilizando o `userdata`.
* Realizar requisições de chat para o modelo do DeepSeek usando o cliente oficial compatível da biblioteca `openai`.
* Compreender a estrutura de uma requisição HTTP direta utilizando a biblioteca `requests` e decodificar respostas no formato JSON.
* Diagnosticar erros comuns de comunicação e formatação de cabeçalhos (headers) em requisições de API.

### Pré-requisitos e Bibliotecas Utilizadas
* **`google.colab.userdata`**: Essencial para recuperar segredos (como chaves de API) configurados no painel lateral do Colab, evitando expor senhas diretamente no código.
* **`openai`**: O DeepSeek possui uma API compatível com o formato da OpenAI. Usamos essa biblioteca por ser uma interface de alto nível simplificada e muito utilizada no mercado.
* **`requests`**: Biblioteca padrão para fazer requisições HTTP manuais. Útil quando queremos entender o que acontece por baixo dos panos sem depender de SDKs específicos.
* **`json`**: Utilizada para formatar nossos dados Python no padrão JSON (JavaScript Object Notation), que é o formato de comunicação universal das APIs web.
```

In [7]:
from google.colab import userdata


```markdown
## Passo 1: Importando o gerenciador de segredos do Colab

### O que este passo faz
Importa o módulo `userdata` do ecossistema do Google Colab.

### Por que ele é necessário
Para que sua aplicação possa conversar com o DeepSeek, ela precisa de uma chave de acesso única (API Key). Escrever essa chave diretamente no código é um risco de segurança. O `userdata` nos permite acessar essa chave de forma segura a partir do gerenciador de segredos do Colab (o ícone de chave 🔑 no menu lateral).
```

In [8]:
api_key = userdata.get('deepseek_api')

```markdown
## Passo 2: Recuperando a Chave de API

### O que este passo faz
Busca no gerenciador de segredos a variável chamada `'deepseek_api'` e a armazena na variável `api_key`.

### Por que ele é necessário
Armazenar a chave em uma variável nos permite reutilizá-la facilmente em diferentes partes do código sem precisar digitar ou expor seu valor real.
```

In [ ]:
from openai import OpenAI
url = 'https://api.deepseek.com'
client = OpenAI(api_key=api_key, base_url=url)

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": "Você é um assistente que deve responder de forma clara e objetiva"},
        {"role": "user", "content": "Cite uma frase de Carl Sagan"}
    ]
)
print(response.choices[0].message.content)

```markdown
## Passo 3: Criando o cliente e gerando texto com a biblioteca OpenAI

### O que este passo faz
Instancia o cliente de conexão configurando a URL base do DeepSeek e envia uma mensagem estruturada (com regras de sistema e a pergunta do usuário) usando o modelo `deepseek-chat`.

### Por que ele é necessário
Como o DeepSeek compartilha o mesmo padrão de comunicação (protocolo) da OpenAI, podemos economizar tempo de desenvolvimento utilizando uma biblioteca consolidada de mercado para realizar a chamada e tratar a resposta de forma limpa.

### Como interpretar o resultado
O modelo deve retornar a resposta à pergunta "Cite uma frase de Carl Sagan" de forma clara e objetiva, conforme instruído na mensagem de sistema. O resultado impresso na tela é o texto direto extraído do primeiro item da lista de escolhas da resposta do modelo (`response.choices[0].message.content`).
```

In [10]:
import requests
import json

url = 'https://api.deepseek.com/chat/completions'

headers = {
    "Content-type" : "aplication/json",
    "Authorization" : f"Bearer {api_key}"
}

payload = {
    "model" : "deepseek-chat",
    "messages" : [
        {"role": "system", "content": "Você é um assistente que deve responder de forma clara e objetiva"},
        {"role": "user", "content": "Cite uma frase de Carl Sagan"}
    ]
}

response = requests.post(url, headers=headers, data=json.dumps(payload))

result = response.json()
print(result['choices'][0]['message']['content'])

KeyError: 'choices'

```markdown
## Passo 4: Chamada Direta via Requisição HTTP (Requests)

### O que este passo faz
Tenta realizar a mesma chamada de chat ao DeepSeek, mas de forma manual, construindo o cabeçalho de autenticação (headers) e o corpo de dados (payload) manualmente para enviar por meio de um método POST HTTP.

### Por que ele é necessário
Entender como fazer chamadas HTTP brutas é fundamental quando você precisa integrar a API em linguagens ou ambientes que não possuem um SDK oficial pronto.

### Como interpretar o resultado
O código busca obter a propriedade `['choices']` do JSON retornado. No entanto, ao executar, este bloco resulta em um erro de execução `KeyError: 'choices'`.

💡 **Para pensar:**
Observe atentamente a definição do dicionário `headers` neste bloco:
```python
headers = {
    "Content-type" : "aplication/json",
    "Authorization" : f"Bearer {api_key}"
}
```
O valor do cabeçalho de tipo de conteúdo está escrito como `"aplication/json"` (com apenas um "p"). APIs de servidores são extremamente exigentes quanto à sintaxe. A falta da grafia correta `"application/json"` faz com que o servidor não entenda o formato dos dados que você enviou, retornando uma mensagem de erro em formato de dicionário sem a chave `"choices"`.

---

## Erros Comuns e Dicas 🛠️
* **`KeyError: 'choices'`**: Geralmente ocorre quando a requisição falha e o servidor nos devolve um JSON contendo detalhes do erro (como chaves de autenticação inválidas ou cabeçalhos mal formatados) em vez do resultado esperado.
* **Exposição de Chaves**: Nunca faça upload de notebooks com sua chave de API escrita no código para repositórios públicos como o GitHub.

## Resumo Final (O que aprendemos) 📝
* O DeepSeek é altamente compatível com a biblioteca padrão da OpenAI, o que facilita muito a migração de projetos.
* O cabeçalho `Content-Type: application/json` é obrigatório em requisições POST para APIs modernas para avisar ao servidor que estamos enviando dados estruturados.
* Gerenciar chaves com `google.colab.userdata` mantém nossas credenciais seguras.

## Exercícios Propostos 🧠
1. **Fácil**: Corrija o erro ortográfico no dicionário `headers` do último bloco de código para que a chamada manual de API funcione perfeitamente.
2. **Médio**: Modifique o parâmetro `messages` do Passo 3 para criar um assistente especializado em traduzir expressões brasileiras para o inglês.
3. **Desafiador**: Altere a chamada do Passo 3 para ajustar o parâmetro `temperature` da API (pesquise na documentação do DeepSeek sobre esse parâmetro) e observe como valores altos (ex: 1.3) ou baixos (ex: 0.2) afetam a criatividade da resposta.

## Para ir além 🚀
* **Modelos do DeepSeek**: Pesquise sobre as diferenças de finalidade entre o modelo de chat genérico (`deepseek-chat`) e o modelo focado em raciocínio lógico e matemática (`deepseek-reasoner`).
* **Variáveis de Temperatura e Max Tokens**: Entenda como esses hiperparâmetros moldam a extensão e a aleatoriedade das respostas de Modelos de Linguagem (LLMs).
```